# Frosted Prison: complete generic / text-guided sweep

This focused notebook generates the missing **14 final images** for the selected
`w01_frosted_prison` material:

- seven start times: `0.02, 0.04, 0.06, 0.10, 0.14, 0.25, 0.50`;
- generic editing with `a high quality photo`;
- text-guided editing with the selected frozen-fortress prompt;
- fixed seed `12`, CFG weight `7`, and Euler step size `0.02`.

Upload `parta_runtime_assets.zip` and `parta_candidate_pack.zip` when prompted.
Run the cells from top to bottom. Every result is saved immediately, so rerunning
the sweep skips completed PNGs. Enable Google Drive persistence if the full run
may span more than one Colab session.

The text conditions are created only through `model.encode(prompt)`. No embedding
tensor is expanded or printed.

In [ ]:
#@title 1. Load the model once
# The two ZIP files stay external so Colab never tries to pretty-print binary data.
import os, sys, time, io, json, math, zipfile
from pathlib import Path
from uuid import uuid4

IN_COLAB = 'google.colab' in sys.modules
WORK = Path('/content/cs180-hw3') if IN_COLAB else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)


def upload_named_archive(name):
    path = Path.cwd() / name
    if path.is_file():
        return path
    if not IN_COLAB:
        raise FileNotFoundError(f'Place {name} beside the notebook and rerun this cell.')
    from google.colab import files
    print(f'Upload {name}.')
    uploaded = files.upload()
    if name not in uploaded:
        raise FileNotFoundError(f'Expected {name}; received {list(uploaded)}')
    return Path.cwd() / name


def safe_extract(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for entry in archive.infolist():
            target = (destination / entry.filename).resolve()
            if destination != target and destination not in target.parents:
                raise ValueError(f'Unsafe archive path: {entry.filename}')
        archive.extractall(destination)


if not (WORK / 'flow_model.py').is_file():
    safe_extract(upload_named_archive('parta_runtime_assets.zip'), WORK)

os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
from colab_setup import configure
configure('parta', install_dependencies=IN_COLAB)

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from torchvision.transforms.functional import gaussian_blur
from flow_model import FlowModel, preprocess, to_pil, MODEL_REVISION, TEXT_REVISION

torch.set_num_threads(4)
MODEL_KEY = (MODEL_REVISION, TEXT_REVISION, 'cuda:0', 'float32')
if globals().get('_loaded_model_key') != MODEL_KEY or 'model' not in globals():
    print('Loading PixNerd. This is the only expensive setup step.')
    model = FlowModel()
    _loaded_model_key = MODEL_KEY
else:
    print('Reusing the PixNerd model already loaded in this runtime.')

predict_velocity = model.predict_velocity
device = model.device
photo_condition = model.encode('a high quality photo')
null_condition = model.encode('')
H, W = 0.02, 7
print('Ready on', torch.cuda.get_device_name(0))


In [ ]:
#@title 2. Upload the candidate pack once
PACK_DIR = WORK / 'candidate_pack'
if not (PACK_DIR / 'experiments.json').is_file():
    PACK_DIR.mkdir(parents=True, exist_ok=True)
    safe_extract(upload_named_archive('parta_candidate_pack.zip'), PACK_DIR)

CONFIG = json.loads((PACK_DIR / 'experiments.json').read_text(encoding='utf-8'))
RESULTS = WORK / 'results' / 'candidate_trials'
RESULTS.mkdir(parents=True, exist_ok=True)
print('Loaded', len(CONFIG['warden_candidates']), 'image candidates,',
      len(CONFIG['anagrams']), 'anagram pairs, and', len(CONFIG['hybrids']), 'hybrid pairs.')
display(Image.open(PACK_DIR / 'warden_candidates_preview.jpg'))


In [ ]:
#@title 3. Sampling helpers (run once)
def load_input(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return preprocess(path).to(device)


def noise(seed):
    generator = torch.Generator().manual_seed(int(seed))
    return torch.randn((1, 3, 512, 512), generator=generator).to(device)


@torch.inference_mode()
def euler_sample(x_start, t_start, step_size, velocity):
    x = x_start.detach().float().clone()
    t = float(t_start)
    while t < 1 - 1e-10:
        step = min(float(step_size), 1 - t)
        x = x + step * velocity(x, t)
        t = min(1.0, t + step)
    return x


@torch.inference_mode()
def cfg_velocity(x, t, condition, w=7):
    conditional = predict_velocity(x, t, condition)
    unconditional = predict_velocity(x, t, null_condition)
    return conditional + float(w) * (conditional - unconditional)


@torch.inference_mode()
def edit(image, epsilon, t_start, condition):
    start = (1 - float(t_start)) * epsilon + float(t_start) * image
    return euler_sample(start, t_start, H,
                        lambda state, t: cfg_velocity(state, t, condition, W))


def rotate180(x):
    return torch.flip(x, (-2, -1))


@torch.inference_mode()
def visual_anagram(epsilon, condition_a, condition_b):
    def velocity(x, t):
        upright = cfg_velocity(x, t, condition_a, W)
        rotated = rotate180(cfg_velocity(rotate180(x), t, condition_b, W))
        return (upright + rotated) / 2
    return euler_sample(epsilon, 0, H, velocity)


def lowpass(x, sigma):
    # 265 is the assignment helper's 512-pixel kernel size; it also covers sigma 12--24.
    return gaussian_blur(x, [265, 265], [float(sigma), float(sigma)])


@torch.inference_mode()
def hybrid_image(epsilon, condition_distant, condition_close, sigma):
    def velocity(x, t):
        distant = cfg_velocity(x, t, condition_distant, W)
        close = cfg_velocity(x, t, condition_close, W)
        return lowpass(distant, sigma) + close - lowpass(close, sigma)
    return euler_sample(epsilon, 0, H, velocity)


def select(kind, experiment_id):
    matches = [item for item in CONFIG[kind] if item['id'] == experiment_id]
    if len(matches) != 1:
        raise KeyError(f'Unknown id {experiment_id!r}; choose one printed in the notebook.')
    return matches[0]


def trial_folder(kind, experiment_id):
    name = time.strftime('%Y%m%d-%H%M%S') + '-' + kind + '-' + experiment_id
    folder = RESULTS / name
    folder.mkdir(parents=True, exist_ok=False)
    return folder


def write_metadata(folder, metadata):
    (folder / 'settings.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print('Saved to', folder)


assert torch.equal(rotate180(rotate180(torch.zeros(1, 3, 4, 4))), torch.zeros(1, 3, 4, 4))
print('Helpers ready. The sweep cell saves each completed image immediately.')

## Run the selected sweep

The original image and every generated result are saved as separate 512 px PNGs.
Two row summaries and a paired comparison sheet are also produced for review.

In [ ]:
#@title 4. Generate all missing Frosted Prison edits
RUN_SWEEP = True #@param {type:"boolean"}
SEED = 12 #@param {type:"integer"}
SAVE_TO_GOOGLE_DRIVE = True #@param {type:"boolean"}
RUN_GENERIC = True #@param {type:"boolean"}
RUN_TEXT_GUIDED = True #@param {type:"boolean"}

STARTS = [0.02, 0.04, 0.06, 0.10, 0.14, 0.25, 0.50]
ITEM_ID = 'w01_frosted_prison'
GENERIC_PROMPT = 'a high quality photo'

if SAVE_TO_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/cs180_pj3_frosted_sweep')
else:
    OUT = WORK / 'results' / 'frosted_sweep'
OUT.mkdir(parents=True, exist_ok=True)


def result_path(mode, t):
    return OUT / f'{mode}_t{t:.2f}.png'


def load_saved(path):
    return preprocess(path).to(device)


if RUN_SWEEP:
    item = select('warden_candidates', ITEM_ID)
    original = load_input(PACK_DIR / 'candidates_512' / f'{ITEM_ID}.jpg')
    to_pil(original).save(OUT / 'original.png')
    epsilon = noise(SEED)

    prompts = {}
    if RUN_GENERIC:
        prompts['generic'] = GENERIC_PROMPT
    if RUN_TEXT_GUIDED:
        prompts['text_guided'] = item['prompt']

    # Encode each prompt once. The embedding is never explicitly printed.
    conditions = {mode: model.encode(prompt) for mode, prompt in prompts.items()}
    results = {mode: {} for mode in prompts}

    for mode, condition in conditions.items():
        print(f'\n=== {mode}: {prompts[mode]} ===')
        for t in STARTS:
            path = result_path(mode, t)
            if path.is_file():
                print(f'skip existing {path.name}')
                result = load_saved(path)
            else:
                print(f'generating {mode}, t={t:.2f}')
                result = edit(original, epsilon, t, condition)
                to_pil(result).save(path)
            results[mode][t] = result
            display(to_pil(result))
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    manifest = {
        'id': ITEM_ID,
        'seed': int(SEED),
        'starts': STARTS,
        'cfg_weight': W,
        'euler_step': H,
        'prompts': prompts,
    }
    (OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

    for mode in prompts:
        fig, axes = plt.subplots(1, 8, figsize=(24, 3.35), layout='constrained')
        views = [('original', original)] + [(f't={t:.2f}', results[mode][t]) for t in STARTS]
        for axis, (label, view) in zip(axes, views):
            axis.imshow(to_pil(view)); axis.set_title(label); axis.axis('off')
        fig.suptitle(f'{ITEM_ID} - {mode} - seed {SEED}\n{prompts[mode]}', wrap=True)
        fig.savefig(OUT / f'{mode}_row.png', dpi=160, bbox_inches='tight')
        plt.show(); plt.close(fig)

    if set(prompts) == {'generic', 'text_guided'}:
        fig, axes = plt.subplots(2, 7, figsize=(21, 6.5), layout='constrained')
        for row, mode in enumerate(('generic', 'text_guided')):
            for col, t in enumerate(STARTS):
                axes[row, col].imshow(to_pil(results[mode][t]))
                axes[row, col].set_title(f'{mode}\nt={t:.2f}')
                axes[row, col].axis('off')
        fig.suptitle(f'{ITEM_ID}: matched generic vs text-guided sweep, seed {SEED}')
        fig.savefig(OUT / 'paired_sweep.png', dpi=160, bbox_inches='tight')
        plt.show(); plt.close(fig)

    print('Saved all available outputs to', OUT)
else:
    print('Set RUN_SWEEP=True when ready. Existing PNGs will be skipped.')

In [ ]:
#@title 5. Download the accumulated outputs
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
if DOWNLOAD_RESULTS:
    import shutil
    archive = shutil.make_archive('/content/frosted_prison_complete_sweep', 'zip', OUT)
    print('Created', archive)
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
else:
    print('Set DOWNLOAD_RESULTS=True when you want the ZIP.')